In [4]:
from __future__ import annotations

import math
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [5]:
DATA_PATH = Path("/mnt/primary/sp500_100_companies_10years.csv")

HORIZONS = [1, 3, 5]
RANDOM_STATE = 42

TRAIN_END = pd.Timestamp("2024-12-31")
VALIDATION_START = pd.Timestamp("2025-01-01")
VALIDATION_END = pd.Timestamp("2025-12-31")
TEST_START = pd.Timestamp("2026-01-01")
TEST_END = pd.Timestamp("2026-07-14")

MODEL_OUTPUT_ROOT = Path("/mnt/primary/Baseline/baseline models/saved_models")
MODEL_SAVE_NAME = "ElasticNet"
MODEL_DIRECTORY = MODEL_OUTPUT_ROOT / MODEL_SAVE_NAME
MODEL_DIRECTORY.mkdir(parents=True, exist_ok=True)

RESULTS_ROOT = Path("/mnt/primary/Baseline/baseline_results")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATA_PATH.resolve())
print("Training period: dataset start to", TRAIN_END.date())
print("Validation period:", VALIDATION_START.date(), "to", VALIDATION_END.date())
print("Testing period:", TEST_START.date(), "to", TEST_END.date())
print("Final saved models: trained on all usable data through", TEST_END.date())


Dataset: /mnt/primary/sp500_100_companies_10years.csv
Training period: dataset start to 2024-12-31
Validation period: 2025-01-01 to 2025-12-31
Testing period: 2026-01-01 to 2026-07-14
Final saved models: trained on all usable data through 2026-07-14


In [6]:
REQUIRED_COLUMNS = {"Date", "Symbol", "Open", "High", "Low", "Close", "Volume"}

def load_market_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path.resolve()}")

    data = pd.read_csv(path)
    missing = REQUIRED_COLUMNS - set(data.columns)

    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    price_column = "Adj Close" if "Adj Close" in data.columns else "Close"
    data["Date"] = pd.to_datetime(data["Date"], errors="coerce").dt.normalize()
    data["Symbol"] = data["Symbol"].astype(str).str.strip().str.upper()

    for column in ["Open", "High", "Low", "Close", "Volume", price_column]:
        data[column] = pd.to_numeric(data[column], errors="coerce")

    data = (
        data.dropna(subset=["Date", "Symbol", "Open", "High", "Low", "Close", "Volume", price_column])
        .drop_duplicates(subset=["Symbol", "Date"], keep="last")
        .sort_values(["Symbol", "Date"])
        .reset_index(drop=True)
    )

    data["AdjustedPrice"] = data[price_column]
    return data

market_data = load_market_data(DATA_PATH)

print("Rows:", f"{len(market_data):,}")
print("Companies:", market_data["Symbol"].nunique())
print("Dataset dates:", market_data["Date"].min().date(), "to", market_data["Date"].max().date())


Rows: 241,058
Companies: 100
Dataset dates: 2016-07-14 to 2026-07-14


In [7]:
def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    return numerator / denominator.replace(0, np.nan)


def calculate_rsi(price: pd.Series, period: int = 14) -> pd.Series:
    delta = price.diff()
    gains = delta.clip(lower=0)
    losses = -delta.clip(upper=0)
    avg_gain = gains.rolling(period, min_periods=period).mean()
    avg_loss = losses.rolling(period, min_periods=period).mean()
    rs = safe_divide(avg_gain, avg_loss)
    rsi = 100 - (100 / (1 + rs))
    rsi = rsi.mask((avg_loss == 0) & (avg_gain > 0), 100)
    rsi = rsi.mask((avg_gain == 0) & (avg_loss > 0), 0)
    return rsi


def engineer_features_for_company(company: pd.DataFrame) -> pd.DataFrame:
    company = company.sort_values("Date").copy()
    price = company["AdjustedPrice"]
    close = company["Close"]
    open_price = company["Open"]
    high = company["High"]
    low = company["Low"]
    volume = company["Volume"].replace(0, np.nan)

    for window in [28, 63, 126]:
        company[f"Return_{window}D"] = price.pct_change(window)
        company[f"Volatility_{window}D"] = price.pct_change().rolling(window, min_periods=window).std()
        rolling_mean = price.rolling(window, min_periods=window).mean()
        company[f"MeanPrice_{window}D"] = rolling_mean
        company[f"Price_to_Mean_{window}D"] = safe_divide(price, rolling_mean) - 1

    for window in [1, 3, 5, 10, 20]:
        company[f"Return_{window}D"] = price.pct_change(window)

    company["Return_1D_Base"] = price.pct_change(1)

    for lag in [1, 2, 3, 5, 10]:
        company[f"Return_1D_Lag_{lag}"] = company["Return_1D_Base"].shift(lag)

    for window in [5, 10, 20, 50, 100]:
        rolling_mean = price.rolling(window, min_periods=window).mean()
        company[f"Price_to_MA_{window}"] = safe_divide(price, rolling_mean) - 1

    for window in [5, 10, 20, 50]:
        company[f"Volatility_{window}D"] = price.pct_change().rolling(window, min_periods=window).std()

    high_20 = price.rolling(20, min_periods=20).max()
    low_20 = price.rolling(20, min_periods=20).min()
    company["Price_to_20D_High"] = safe_divide(price, high_20) - 1
    company["Price_to_20D_Low"] = safe_divide(price, low_20) - 1

    company["Intraday_Return"] = safe_divide(close - open_price, open_price)
    company["High_Low_Range"] = safe_divide(high - low, open_price)
    company["Close_Position_In_Range"] = safe_divide(close - low, high - low)
    company["Overnight_Gap"] = safe_divide(open_price, close.shift(1)) - 1
    company["Volume_Change_1D"] = volume.pct_change(1)
    company["Volume_Change_5D"] = volume.pct_change(5)

    for window in [5, 20, 50]:
        company[f"Relative_Volume_{window}"] = safe_divide(volume, volume.rolling(window, min_periods=window).mean())

    company["RSI_14"] = calculate_rsi(price, 14) / 100.0

    ema_12 = price.ewm(span=12, adjust=False).mean()
    ema_26 = price.ewm(span=26, adjust=False).mean()
    macd = ema_12 - ema_26
    signal = macd.ewm(span=9, adjust=False).mean()
    company["MACD_Relative"] = safe_divide(macd, price)
    company["MACD_Signal_Relative"] = safe_divide(signal, price)
    company["MACD_Histogram_Relative"] = safe_divide(macd - signal, price)

    ma_20 = price.rolling(20, min_periods=20).mean()
    std_20 = price.rolling(20, min_periods=20).std()
    upper_band = ma_20 + (2 * std_20)
    lower_band = ma_20 - (2 * std_20)
    company["Bollinger_Width"] = safe_divide(upper_band - lower_band, ma_20)
    company["Bollinger_Position"] = safe_divide(price - lower_band, upper_band - lower_band)

    previous_close = close.shift(1)
    true_range = pd.concat(
        [high - low, (high - previous_close).abs(), (low - previous_close).abs()],
        axis=1,
    ).max(axis=1)

    company["ATR_14_Relative"] = safe_divide(true_range.rolling(14, min_periods=14).mean(), price)
    company["Day_Of_Week"] = company["Date"].dt.dayofweek
    company["Month"] = company["Date"].dt.month
    company["Quarter"] = company["Date"].dt.quarter

    for horizon in HORIZONS:
        future_price = price.shift(-horizon)
        company[f"FuturePrice_{horizon}D"] = future_price
        company[f"TargetReturn_{horizon}D"] = safe_divide(future_price, price) - 1

    return company


BASIC_FEATURES = [
    "Return_28D", "Return_63D", "Return_126D",
    "Volatility_28D", "Volatility_63D", "Volatility_126D",
    "MeanPrice_28D", "MeanPrice_63D", "MeanPrice_126D",
    "Price_to_Mean_28D", "Price_to_Mean_63D", "Price_to_Mean_126D",
]

ADVANCED_FEATURES = BASIC_FEATURES + [
    "Return_1D", "Return_3D", "Return_5D", "Return_10D", "Return_20D",
    "Return_1D_Lag_1", "Return_1D_Lag_2", "Return_1D_Lag_3", "Return_1D_Lag_5", "Return_1D_Lag_10",
    "Price_to_MA_5", "Price_to_MA_10", "Price_to_MA_20", "Price_to_MA_50", "Price_to_MA_100",
    "Volatility_5D", "Volatility_10D", "Volatility_20D", "Volatility_50D",
    "Price_to_20D_High", "Price_to_20D_Low", "Intraday_Return", "High_Low_Range",
    "Close_Position_In_Range", "Overnight_Gap", "Volume_Change_1D", "Volume_Change_5D",
    "Relative_Volume_5", "Relative_Volume_20", "Relative_Volume_50", "RSI_14",
    "MACD_Relative", "MACD_Signal_Relative", "MACD_Histogram_Relative",
    "Bollinger_Width", "Bollinger_Position", "ATR_14_Relative", "Day_Of_Week", "Month", "Quarter",
]

FEATURE_SETS = {"Basic": BASIC_FEATURES, "Advanced": ADVANCED_FEATURES}

featured_data = (
    market_data.groupby("Symbol", group_keys=False)
    .apply(engineer_features_for_company)
    .reset_index(drop=True)
    .replace([np.inf, -np.inf], np.nan)
)

print("Basic features:", len(BASIC_FEATURES))
print("Advanced features:", len(ADVANCED_FEATURES))


Basic features: 12
Advanced features: 52


In [8]:
MODEL_DISPLAY_NAME = "Elastic Net"

ELASTICNET_ALPHA = 0.001
ELASTICNET_L1_RATIO = 0.50

def create_model() -> Pipeline:
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", ElasticNet(alpha=ELASTICNET_ALPHA, l1_ratio=ELASTICNET_L1_RATIO, max_iter=10000, random_state=RANDOM_STATE)),
    ])

In [9]:
def rmse(actual_price: np.ndarray, predicted_price: np.ndarray) -> float:
    return math.sqrt(mean_squared_error(actual_price, predicted_price))


result_rows = []
symbols = sorted(featured_data["Symbol"].unique())

for symbol_index, symbol in enumerate(symbols, start=1):
    company_all = (featured_data[featured_data["Symbol"] == symbol].sort_values("Date").copy())

    company_model_package = {
        "Symbol": symbol,
        "Algorithm": MODEL_DISPLAY_NAME,
        "ModelSaveName": MODEL_SAVE_NAME,
        "FeatureSets": {"Basic": {}, "Advanced": {}},
        "FeatureColumns": {"Basic": BASIC_FEATURES, "Advanced": ADVANCED_FEATURES},
        "Horizons": HORIZONS,
        "TargetDefinition": "Future return = FutureAdjustedPrice / CurrentAdjustedPrice - 1",
        "DataPath": str(DATA_PATH),
        "FinalFit": "All usable observations through 2026-07-14",
    }

    print(f"[{symbol_index}/{len(symbols)}] {symbol}")

    for feature_set_name, feature_columns in FEATURE_SETS.items():
        for horizon in HORIZONS:
            target_column = f"TargetReturn_{horizon}D"
            future_price_column = f"FuturePrice_{horizon}D"
            required_columns = [*feature_columns, target_column, future_price_column, "AdjustedPrice"]

            company = company_all.dropna(subset=required_columns).copy()

            train = company[company["Date"] <= TRAIN_END].copy()
            validation = company[
                (company["Date"] >= VALIDATION_START)
                & (company["Date"] <= VALIDATION_END)
            ].copy()
            test = company[
                (company["Date"] >= TEST_START)
                & (company["Date"] <= TEST_END)
            ].copy()

            if train.empty or validation.empty or test.empty:
                print(f"  Skipping {feature_set_name} {horizon}D: incomplete chronological split.")
                continue

            evaluation_model = create_model()
            evaluation_model.fit(train[feature_columns], train[target_column])

            validation_return = evaluation_model.predict(validation[feature_columns])
            validation_price = validation["AdjustedPrice"].to_numpy() * (1 + validation_return)
            validation_rmse = rmse(validation[future_price_column].to_numpy(), validation_price)

            train_validation = pd.concat([train, validation], ignore_index=True)
            test_model = create_model()
            test_model.fit(train_validation[feature_columns], train_validation[target_column])

            test_return = test_model.predict(test[feature_columns])
            test_price = test["AdjustedPrice"].to_numpy() * (1 + test_return)
            test_rmse = rmse(test[future_price_column].to_numpy(), test_price)

            result_rows.append(
                {
                    "ML_Model": MODEL_DISPLAY_NAME,
                    "Symbol": symbol,
                    "FeatureSet": feature_set_name,
                    "Horizon_Days": horizon,
                    "Validation_RMSE": validation_rmse,
                    "Test_RMSE": test_rmse,
                }
            )

            final_model = create_model()
            final_model.fit(company[feature_columns], company[target_column])
            company_model_package["FeatureSets"][feature_set_name][horizon] = final_model

    if any(company_model_package["FeatureSets"][name] for name in FEATURE_SETS):
        joblib.dump(company_model_package, MODEL_DIRECTORY / f"{symbol}.joblib")

results_df = pd.DataFrame(result_rows)

if results_df.empty:
    raise ValueError("No model results were produced.")

print("Evaluation complete.")
print("Companies evaluated:", results_df["Symbol"].nunique())
print("Saved model directory:", MODEL_DIRECTORY.resolve())


[1/100] AAPL
[2/100] ABNB
[3/100] ACN
[4/100] ADBE
[5/100] AEP
[6/100] AES
[7/100] AFL
[8/100] AIG
[9/100] AMZN
[10/100] AOS
[11/100] APA
[12/100] APD
[13/100] APO
[14/100] AXP
[15/100] BA
[16/100] BKNG
[17/100] BKR
[18/100] BR
[19/100] CARR
[20/100] CASY
[21/100] CAT
[22/100] CBRE
[23/100] CEG
[24/100] CHRW
[25/100] CI
[26/100] CL
[27/100] COO
[28/100] COP
[29/100] COST
[30/100] CSCO
[31/100] CSGP
[32/100] CTAS
[33/100] CTSH
[34/100] CTVA
[35/100] CVS
[36/100] CVX
[37/100] D
[38/100] DASH
[39/100] DG
[40/100] DHR
[41/100] DIS
[42/100] DVA
[43/100] DVN
[44/100] DXCM
[45/100] EA
[46/100] ED
[47/100] EOG
[48/100] EW
[49/100] EXE
[50/100] FANG
[51/100] GEHC
[52/100] GILD
[53/100] GIS
[54/100] GOOGL
[55/100] GS
[56/100] HPE
[57/100] IFF
[58/100] IP
[59/100] IRM
[60/100] JPM
[61/100] KIM
[62/100] KO
[63/100] LNT
[64/100] LYB
[65/100] MAA
[66/100] META
[67/100] MLM
[68/100] MMM
[69/100] MOS
[70/100] NFLX
[71/100] NKE
[72/100] NOW
[73/100] OMC
[74/100] PEP
[75/100] PYPL
[76/100] RL
[77/100] S

In [10]:
summary_results = (
    results_df.groupby(["ML_Model", "FeatureSet", "Horizon_Days"], as_index=False)[["Validation_RMSE", "Test_RMSE"]]
    .mean()
    .sort_values(["FeatureSet", "Horizon_Days"])
    .reset_index(drop=True)
)

display(summary_results.round(4))

results_df.to_csv(RESULTS_ROOT / f"{MODEL_SAVE_NAME}_company_rmse.csv", index=False)
summary_results.to_csv(RESULTS_ROOT / f"{MODEL_SAVE_NAME}_summary_rmse.csv", index=False)

print("Saved RMSE results to:", RESULTS_ROOT.resolve())


,ML_Model,FeatureSet,Horizon_Days,Validation_RMSE,Test_RMSE
0,Elastic Net,Advanced,1,3.0924,3.8925
1,Elastic Net,Advanced,3,5.3929,6.6705
2,Elastic Net,Advanced,5,6.8759,8.6884
3,Elastic Net,Basic,1,3.0688,3.8800
4,Elastic Net,Basic,3,5.3001,6.6037
5,Elastic Net,Basic,5,6.7684,8.5238


Saved RMSE results to: /mnt/primary/Baseline/baseline_results
